# Week Ten | Document Classification
**Sachi Kapoor · DATA 620 Web Analytics**

Can a model learn to separate spam from legitimate email using patterns in previously labeled messages? I use [UCI Spambase](https://doi.org/10.24432/C53G6X), which represents each email with 57 numeric features. These include selected word and character frequencies and statistics about runs of capital letters. The target is 1 for spam and 0 for ham (legitimate mail).

This notebook uses pandas and scikit-learn. It downloads the dataset from UCI when run. The supplied features are already computed from the emails.

In [1]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

url = "https://archive.ics.uci.edu/static/public/94/spambase.zip"
with urlopen(url, timeout=30) as response:
    archive_bytes = response.read()

with ZipFile(BytesIO(archive_bytes)) as archive:
    metadata = archive.read("spambase.names").decode("latin-1")
    feature_names = [
        line.split(":", 1)[0].strip()
        for line in metadata.splitlines()
        if line.strip().endswith("continuous.")
    ]
    with archive.open("spambase.data") as data:
        emails = pd.read_csv(data, header=None, na_values="?")

assert len(feature_names) == 57 and emails.shape[1] == 58
emails.columns = feature_names + ["spam"]
assert set(emails["spam"]) == {0, 1}

print("Original rows:", len(emails))
print("Missing feature values:", emails[feature_names].isna().sum().sum())
print("Identical feature-and-label rows removed:", emails.duplicated().sum())
emails = emails.drop_duplicates()
print(emails["spam"].map({0: "Ham", 1: "Spam"}).value_counts().to_string())

Original rows: 4601
Missing feature values: 0
Identical feature-and-label rows removed: 391
spam
Ham     2531
Spam    1679


## Train on labeled email

I remove identical feature-and-label rows before splitting so they cannot inflate the test score by appearing in both sets. I reserve 20% of the remaining rows for testing and stratify the split to keep the spam-to-ham ratio similar. My comparison is a majority-class baseline that always predicts the most frequent training label.

I fit logistic regression because it provides a direct way to see whether the supplied numeric features help predict the two classes. Median imputation and standardization are fitted in the training pipeline, so test values do not influence those transformations.

In [2]:
X = emails[feature_names]
y = emails["spam"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

classifier = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)
classifier.fit(X_train, y_train)

Training rows: 3368 | Test rows: 842


## Predict and evaluate on held-out email

I use accuracy to compare with the baseline. For spam specifically, precision answers how often a spam flag is correct, and recall answers how much of the actual spam the classifier catches.

In [3]:
predicted = classifier.predict(X_test)
baseline_accuracy = baseline.score(X_test, y_test)
accuracy = classifier.score(X_test, y_test)
print(f"Majority baseline accuracy: {baseline_accuracy:.1%}")
print(f"Logistic regression accuracy: {accuracy:.1%}")
print()
print(classification_report(y_test, predicted, labels=[0, 1],
                            target_names=["Ham", "Spam"], digits=3))
matrix = confusion_matrix(y_test, predicted, labels=[0, 1])
print(pd.DataFrame(matrix, index=["Actual ham", "Actual spam"],
                   columns=["Predicted ham", "Predicted spam"]).to_string())

print("\nExample predictions from the held-out set")
examples = pd.DataFrame({"Actual": y_test.to_numpy(), "Predicted": predicted})
print(examples.head(8).replace({0: "Ham", 1: "Spam"}).to_string(index=False))

Majority baseline accuracy: 60.1%
Logistic regression accuracy: 93.5%

              precision    recall  f1-score   support

         Ham      0.938     0.955     0.946       506
        Spam      0.930     0.905     0.917       336

    accuracy                          0.935       842
   macro avg      0.934     0.930     0.932       842
weighted avg      0.935     0.935     0.935       842

             Predicted ham  Predicted spam
Actual ham             483              23
Actual spam             32             304

Example predictions from the held-out set
Actual Predicted
  Spam      Spam
   Ham       Ham
  Spam      Spam
  Spam       Ham
   Ham      Spam
   Ham       Ham
  Spam       Ham
  Spam      Spam


## Findings

Logistic regression correctly classified **787 of 842** held-out emails (**93.5% accuracy**), compared with **60.1%** for the majority baseline. It caught **304 of 336** spam emails (**90.5% recall**), and **93.0%** of its spam flags were correct.

The model missed **32 spam emails** and flagged **23 legitimate emails** as spam. Both errors matter, but flagging a legitimate email can make someone miss an important message. I would pay particular attention to those false positives when evaluating a spam filter for use in an inbox.

This test measures performance on held-out records from the same historical corpus. UCI notes that some legitimate-mail indicators, such as “george” and “650,” reflect the people and location represented in its data. Performance could differ on email from another inbox or time period.

**Reference**  
Hopkins, M., Reeber, E., Forman, G., and Suermondt, J. (1999). [Spambase](https://doi.org/10.24432/C53G6X). UCI Machine Learning Repository.